# Morales2018

https://doi.org/10.1111/pce.13119

This notebook demonstrates example analyses using the Morales et al. (2018) dynamic photosynthesis model implemented in `mxlmodels`. The examples are inspired by analyses in the publication; they are not intended as exact reproductions of the published figures.

In [ ]:
import matplotlib.pyplot as plt
from mxlpy import Simulator, make_protocol

from mxlmodels import get_morales2018

## Light input

In the model, `PAR` is a derived quantity defined as `Ib + Ig + Ir`. It therefore cannot be changed directly in a simulation protocol. The protocol instead changes the blue (`Ib`), green (`Ig`), and red (`Ir`) light-input parameters. The model's default spectral composition is retained here: 10% blue, 0% green, and 90% red.

In [ ]:
def light_input(par):
    """Return Ib, Ig, and Ir for a target PAR at the default 10:0:90 spectrum."""
    return {"Ib": 0.10 * par, "Ig": 0.0, "Ir": 0.90 * par}

## Load model

In [ ]:
model = get_morales2018()
model

## Figure 5-style analysis: NPQ components

Morales et al. (2018) decomposed non-photochemical quenching into components associated with energy-dependent quenching (`qE`), chloroplast movement (`qM`), and photoinhibition (`qI`). Here, the model is simulated for 5 min at low irradiance followed by 60 min at high irradiance.

In [ ]:
model = get_morales2018()
sim = Simulator(model)
sim.simulate_protocol(
    protocol=make_protocol(
        [
            (300, light_input(50)),
            (3600, light_input(1000)),
        ]
    ),
    time_points_per_step=500,
)

res = sim.get_result().unwrap_or_err().get_combined()
res.head()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(8, 6), sharex=True)

for ax, var in zip(axs.flat, ["NPQ", "qE", "qM", "qI"]):
    ax.plot(res.index / 60, res[var])
    ax.axvline(5, color="0.5", linestyle="--", linewidth=1)
    ax.set_title(var)
    ax.set_ylabel(var)

for ax in axs[-1, :]:
    ax.set_xlabel("Time / min")

plt.tight_layout()
plt.show()

## Photosynthetic outputs

The next example inspects selected photosynthetic readouts during the same irradiance transition. Separate axes are used because these quantities have different scales and units.

In [ ]:
selected_outputs = ["A", "PhiII", "gsw", "Vc", "Vr", "VrJ"]
fig, axs = plt.subplots(2, 3, figsize=(10, 6), sharex=True)

for ax, var in zip(axs.flat, selected_outputs):
    ax.plot(res.index / 60, res[var])
    ax.axvline(5, color="0.5", linestyle="--", linewidth=1)
    ax.set_title(var)
    ax.set_ylabel(var)

for ax in axs[-1, :]:
    ax.set_xlabel("Time / min")

plt.tight_layout()
plt.show()

## Lightfleck-style response

This example simulates 90 s at low irradiance followed by a 90 s high-light period, similar to the lightfleck analyses discussed by Morales et al. (2018).

In [ ]:
model = get_morales2018()
sim = Simulator(model)
sim.simulate_protocol(
    protocol=make_protocol(
        [
            (90, light_input(50)),
            (90, light_input(1000)),
        ]
    ),
    time_points_per_step=300,
)

fleck = sim.get_result().unwrap_or_err().get_combined()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(7, 8), sharex=True)

for ax, var in zip(axs, ["A", "fRuBP", "fRB"]):
    ax.plot(fleck.index, fleck[var])
    ax.axvline(90, color="0.5", linestyle="--", linewidth=1)
    ax.set_ylabel(var)

axs[-1].set_xlabel("Time / s")
plt.tight_layout()
plt.show()